# Objetivo

Desenvolvimento do Teleutils em ambiente de teste local

# Configuração do Ambiente

## Bibliotecas

In [ ]:
# 1. Módulos da Biblioteca Padrão
import logging
from pathlib import Path

# 2. Módulos de Terceiros
import pandas as pd

from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.window import Window
from pyspark.sql.functions import pandas_udf
from pyspark.sql import types as T # ArrayType, StructType, StructField, StringType, BooleanType

# 3. Teletools/Teleutils
from teleutils.core.extractors import CDRParquetExtractor, CDRTextExtractor
from teleutils.core.transformers import CDRTransformer

In [ ]:
# Configuração do Pandas
# Exibir todas as colunas
pd.set_option("display.max_columns", None)

# Exibir todas as linhas
pd.set_option("display.max_rows", None)

# Evitar que o texto nas células seja cortado
pd.set_option("display.max_colwidth", None)

In [ ]:
# Configuração para exibir logs no output da célula
logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s | %(levelname)s | %(name)s | %(message)s",
    datefmt="%H:%M:%S",
)

# Reduzir ruído de bibliotecas externas (pyspark, py4j, etc.)
logging.getLogger("py4j").setLevel(logging.WARNING)
logging.getLogger("pyspark").setLevel(logging.WARNING)

## Spark

In [ ]:
spark = (
    SparkSession
        .builder
        .appName("SessaoLocal")
        .master("local[12]")
        .config("spark.driver.memory", "24g")
        .getOrCreate()
)
spark

## Arquivos de dados

### Dados de origem

In [ ]:
folder_tim = Path("/assets/monitoring/cdr/cdr_bruto/Semana101/Tim/FIXA/NGN_Huawei")
files_tim = [str(file) for file in folder_tim.glob("*.gz")]

### Arquivos intermediário e final

In [ ]:
extracted_file = "/assets/data/cdr/teletools/db/extraidos/tim_ngn_huawei_extraido.parquet"
transformed_file = "/assets/data/cdr/teletools/db/transformados/tim_ngn_huawei_transformado.parquet"

# Análise

## Dados brutos

In [ ]:
df_raw = spark.read.csv(files_tim, sep=";")
df_raw.limit(5).toPandas()

In [ ]:
column_indices = (0,1,2,3,4,5,6,7,8,9,10,11,12,13,14,15,16,17,18,19,20,21,22,23,24)
column_names = ("recCode","origDate","origTime","discDate","discTime","callDur","origClli","origMem","termClli","termMen","clgNum","dialNum","freeIndicator","answer","compCD","dataCall","satInd","seqNum","nodeId","destNum","ccd","cct","ttot","cpc","outpNum")
columns_to_keep = [
 F.col(df_raw.columns[index]).alias(column_name)
 for index, column_name in zip(
 column_indices,
 column_names,
 )
]
df = df_raw.select(*columns_to_keep)

df.limit(5).toPandas()

In [ ]:
# column_indices = (1,2,3,4,29,30)
# column_names = ("_dia", "_hora", "_dia_fim", "_hora_fim", "_data_hora", "_data_hora_fim")
# columns_to_keep = [
#  F.col(df_raw.columns[index]).alias(column_name)
#  for index, column_name in zip(
#  column_indices,
#  column_names,
#  )
# ]
# df = df_raw.select(*columns_to_keep)

# df.limit(5).toPandas()

In [ ]:
# # 1. Converter a string _data_hora original para o tipo Timestamp
# df = df.withColumn("ts_data_hora", F.to_timestamp("_data_hora", "yyMMddHHmmss")) \
#        .withColumn("ts_data_hora_fim", F.to_timestamp("_data_hora_fim", "yyMMddHHmmss"))

# # 2. Extrair o ano da data (ex: 2026)
# df = df.withColumn("ano", F.year("ts_data_hora"))

# # 3. Converter os campos Juliano (_dia e _hora) em Timestamp
# # Nota: F.make_date exige Spark 3.0+. Para Spark 2.x, use F.to_date(F.concat(F.col("ano"), F.lit("-01-01")))
# df = df.withColumn(
#     "ts_juliano_inicio",
#     F.to_timestamp(
#         F.unix_timestamp(F.make_date(F.col("ano"), F.lit(1), F.lit(1))) 
#         + (F.col("_dia") - 1) * 86400 
#         + F.col("_hora") - 1
#     )
# ).withColumn(
#     "ts_juliano_fim",
#     F.to_timestamp(
#         F.unix_timestamp(F.make_date(F.col("ano"), F.lit(1), F.lit(1))) 
#         + (F.col("_dia_fim") - 1) * 86400 
#         + F.col("_hora_fim") - 1
#     )
# )


# # 5. Comparar as datas (diferença em segundos e validação booleana)
# df = df.withColumn(
#     "datas_inicio_iguais", 
#     F.col("ts_data_hora") == F.col("ts_juliano_inicio")
# ).withColumn(
#     "datas_fim_iguais", 
#     F.col("ts_data_hora_fim") == F.col("ts_juliano_fim")
# )


# df.groupBy("datas_inicio_iguais").pivot("datas_fim_iguais").count().show()

In [ ]:
# column_indices = (1,2,3,4,5,7,9,10,11,17,29,30)
# column_names = ("_data","_hora", "_data_fim", "_hora_fim", "duracao", "rota_entrada","rota_saida","numero_origem","numero_destino","referencia","data_hora", "data_hora_fim")
# columns_to_keep = [
#  F.col(df_raw.columns[index]).alias(column_name)
#  for index, column_name in zip(
#  column_indices,
#  column_names,
#  )
# ]
# df = df_raw.select(*columns_to_keep)

# df.limit(5).toPandas()

In [ ]:
# df.groupBy("rota_entrada").pivot("rota_saida").count().show()

In [ ]:
# column_indices = (5,7,9,10,11,17,29,30)
# column_names = ("duracao","rota_entrada","rota_saida","numero_origem","numero_destino","referencia","data_hora", "data_hora_fim")
# columns_to_keep = [
#  F.col(df_raw.columns[index]).alias(column_name)
#  for index, column_name in zip(
#  column_indices,
#  column_names,
#  )
# ]
# df = df_raw.select(*columns_to_keep)

# df.limit(5).toPandas()

## Extração

In [ ]:
%%time
extractor = CDRTextExtractor(spark)
extractor.extract(files_tim, extracted_file, "stfc_huawei_ngn_tim")

In [ ]:
df_extracted = spark.read.parquet(extracted_file)
df_extracted.show(5,truncate=False)

In [ ]:
df_extracted.withColumn("bilhetador", F.split(F.col("arquivo_origem"),"\.").getItem(0)).show(5)

In [ ]:
df_extracted = df_extracted.withColumn("data_hora", F.to_timestamp("data_hora", "yyMMddHHmmss")).withColumn("data_hora_fim", F.to_timestamp("data_hora_fim", "yyMMddHHmmss"))
df_extracted.show()

In [ ]:
df_extracted.groupBy("data_hora").count().orderBy(F.desc("data_hora")).show(5)

## Transformação

In [ ]:
transformer = CDRTransformer(spark)
transformer.transform_stfc_huawei_ngn_tim(extracted_file, transformed_file)

In [ ]:
df_transformed = spark.read.parquet(transformed_file)
df_transformed.limit(5).toPandas().dropna(how="all",axis=1)